# Kinisot from Python

The same calculation as the command line, through `kinisot.compute_kie()`. Parse each Gaussian output once with `parse_gaussian()`, then reuse the `HessianInput` objects for every substitution and temperature. Run this notebook from the `examples/` directory of a repository checkout (it reads `../tests/data/gaussian`).

In [1]:
import os

from kinisot import compute_kie, parse_gaussian

DATA = os.path.join("..", "tests", "data", "gaussian")
gs = parse_gaussian(os.path.join(DATA, "claisen_gs.out"))
ts = parse_gaussian(os.path.join(DATA, "claisen_ts.out"))
gs.natoms, gs.level_of_theory, gs.symbols[:6]

(14, 'RB3LYP/6-31G(d)', ('C', 'C', 'O', 'C', 'C', 'C'))

## Position scan

One `compute_kie()` call per substituted atom. The result object carries the final numbers (`kie`, `kie_tunnel`, ...) and the per-species factors.

In [2]:
print("%-8s %10s %10s %10s %10s %10s" % ("atoms", "V-ratio", "ZPE", "TRPF", "KIE", "corr-KIE"))
for atoms in ["1", "2", "3", "4", "5", "6", "7,8"]:
    r = compute_kie(rct=gs, ts=ts, iso=atoms, temperature=393.0, scale=0.961)
    print("%-8s %10.4f %10.4f %10.4f %10.4f %10.4f" % (atoms, r.imag_ratio, r.zpe, r.trpf, r.kie, r.kie_tunnel))

atoms       V-ratio        ZPE       TRPF        KIE   corr-KIE
1            1.0079     1.0040     0.9975     1.0127     1.0147
2            1.0003     0.9993     1.0022     1.0019     1.0020
3            1.0070     1.0223     0.9839     1.0188     1.0206
4            1.0127     1.0366     0.9790     1.0297     1.0330
5            1.0002     0.9991     1.0020     1.0019     1.0019
6            1.0081     1.0045     0.9980     1.0148     1.0168
7,8          1.0073     0.8856     1.0061     0.9549     0.9566


## Temperature scan and tunnelling models

In [3]:
print("%8s %10s %10s %10s" % ("T / K", "KIE", "Bell", "Wigner"))
for temperature in range(300, 501, 50):
    bell = compute_kie(rct=gs, ts=ts, iso="4", temperature=temperature, scale=0.961)
    wigner = compute_kie(rct=gs, ts=ts, iso="4", temperature=temperature, scale=0.961, tunneling="wigner")
    print("%8.0f %10.4f %10.4f %10.4f" % (temperature, bell.kie, bell.kie_tunnel, wigner.kie_tunnel))

   T / K        KIE       Bell     Wigner
     300     1.0404     1.0463     1.0449
     350     1.0338     1.0380     1.0372
     400     1.0292     1.0323     1.0319
     450     1.0258     1.0283     1.0280
     500     1.0233     1.0253     1.0251


## Everything the CLI prints, as data

`to_dict()` (and `to_json()`) give the full result; `summary_row()` is the one-line CSV form.

In [4]:
r = compute_kie(rct=gs, ts=ts, iso="4", temperature=393.0, scale=0.961)
print("reaction coordinate: %.1fi / %.1fi cm-1" % (r.other.light.imaginary, r.other.heavy.imaginary))
print("TS external modes:", ["%.1f" % f for f in r.other.light.species[0].discarded])
r.summary_row()

reaction coordinate: 463.9i / 458.1i cm-1
TS external modes: ['-9.7', '-0.2', '0.1', '0.1', '13.3', '17.0']


{'kind': 'KIE',
 'reactant': '../tests/data/gaussian/claisen_gs.out',
 'transition_structure_or_product': '../tests/data/gaussian/claisen_ts.out',
 'labels': '4;4',
 'temperature': 393.0,
 'scale_factor': 0.961,
 'tunneling': 'bell',
 'imag_light': 463.90196697099447,
 'imag_heavy': 458.0769617245422,
 'imag_ratio': 1.0127162152502118,
 'zpe': 1.036593731391347,
 'exc': 1.001969535713699,
 'trpf': 0.9789887482179481,
 'kie': 1.0297423153497844,
 'tunnel_corr': 1.0031569707584038,
 'kie_tunnel': 1.0329931817280347}